In [4]:
import requests
import pandas as pd
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'requests'

In [ ]:
API_KEY = "8de7a4f954a04b1481cf1e583f1decfc"
url = "https://newsapi.org/v2/everything"

params = {
    "q": "financial markets OR stocks OR economy",
    "language": "en",
    "sortBy": "publishedAt",
    "pageSize": 100,
    "apiKey": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

# Parse JSON into DataFrame
articles = [
    {
        "Date": item["publishedAt"],
        "News": item["title"] + " " + (item["description"] or "")
    }
    for item in data["articles"]
]

df = pd.DataFrame(articles)

In [ ]:
df.dropna(inplace=True)
df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
def get_subjectivity(text):
    return TextBlob(text).sentiment.subjectivity

def get_polarity(text):
    return TextBlob(text).sentiment.polarity

df["Subjectivity"] = df["News"].apply(get_subjectivity)
df["Polarity"] = df["News"].apply(get_polarity)

def classify_sentiment(score):
    if score < 0: return "Negative"
    elif score > 0: return "Positive"
    else: return "Neutral"

df["Sentiment"] = df["Polarity"].apply(classify_sentiment)

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(x="Sentiment", data=df)
plt.title("Sentiment Distribution in Google News Articles")
plt.show()

In [ ]:
sentiment_over_time = (
    df.groupby(df["Date"].dt.date)["Sentiment"]
      .value_counts()
      .unstack()
      .fillna(0)
)

plt.figure(figsize=(14,6))
sentiment_over_time.plot(marker="o")
plt.title("Financial News Sentiment Over Time")
plt.xlabel("Date")
plt.ylabel("Article Count")
plt.show()

In [ ]:
summary = df['Sentiment'].value_counts(normalize=True) * 100
print(summary)